**1ST REVIEW**

**TITLE**

IMAGE FORGERY DETECTION

**AIM**

To design and implement an image forensics system that can detect and localize image tampering by identifying copy–move forgery, splicing, and frequency-based manipulation using classical image processing techniques.

**DOMAIN**

Computer Vision / Digital Image Forensics / Image Processing

**PROBLEM STATEMENT**

With the rapid spread of digital images across social media, news platforms, and legal evidence, image manipulation has become easy and widespread. Forged images can mislead the public, spread misinformation, and compromise digital trust.
The problem is to automatically detect whether a given image is authentic or tampered and to highlight the tampered regions, without relying on deep learning or large labeled datasets.

**DATA SOURCE**

User-provided images uploaded directly through Google Colab

Publicly available tampered images (copy–move and splicing forgeries)

Synthetic tampered images created using basic copy–paste operations

📌 No fixed dataset is required, making the system general and flexible.

**EXPECTED OUTCOME**

The system should:

Correctly classify images as TAMPERED or AUTHENTIC

Generate a tamper score indicating confidence

Highlight suspicious regions using:

Binary mask

Color-coded heatmap

Tampered images with clear copy–move or splicing artifacts should be reliably detected.

**2ND REVIEW**

**PROPOSED METHODOLOGY**

The system follows a multi-stage classical image forensics pipeline:

Image Input

User uploads one or more images using the Colab file upload interface.

Preprocessing

Convert image to grayscale

Apply Gaussian blur to reduce noise and enhance structural patterns

Copy–Move Detection

Detect repeated textures and duplicated regions using feature-based analysis.

Splicing Detection

Identify abnormal edges and boundary inconsistencies caused by pasted objects.

Frequency / Wavelet Analysis

Detect high-frequency artifacts introduced during manipulation using Laplacian-based analysis.

Score Fusion

Combine outputs from all detectors using weighted pixel-level scoring.

Decision Making

If tamper score > 0.25 → TAMPERED

Else → AUTHENTIC

Tampered Region Localization

Generate a binary mask of suspicious pixels.

Create a heatmap overlay to visually highlight tampered regions.

IMPORTING ALL LIBRARIES

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim
import pywt
import os
from google.colab import files


DISPLAY IMAGE IN COLAB

In [ ]:
def show_image(title, img):
    plt.figure(figsize=(6,6))
    if len(img.shape) == 2:
        plt.imshow(img, cmap='gray')
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()


preprocessing

In [ ]:
def preprocess(img):
    # Only resize + light sharpening
    h, w = img.shape[:2]
    max_dim = 512

    if max(h, w) > max_dim:
        scale = max_dim / max(h, w)
        img = cv2.resize(img, (int(w*scale), int(h*scale)))

    # Light sharpening (keeps tampering details)
    kernel = np.array([[0,-1,0],
                       [-1,5,-1],
                       [0,-1,0]])
    img = cv2.filter2D(img, -1, kernel)

    return img


IMAGE LOADING FUNCTION


In [ ]:
def load_image(path):
    if not os.path.exists(path):
        return None
    return cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)


**3RD REVIEW**

COPY-MOVE FORGERY DETECTION


In [ ]:
def detect_copy_move(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    orb = cv2.ORB_create(nfeatures=1500)
    kp, des = orb.detectAndCompute(gray, None)

    mask = np.zeros_like(gray)

    if des is None or len(kp) < 10:
        return mask

    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = bf.match(des, des)

    for m in matches:
        if m.queryIdx == m.trainIdx:
            continue
        if m.distance < 30:
            p1 = kp[m.queryIdx].pt
            p2 = kp[m.trainIdx].pt
            if abs(p1[0] - p2[0]) + abs(p1[1] - p2[1]) < 12:
                continue
            cv2.circle(mask, (int(p1[0]), int(p1[1])), 5, 255, -1)
            cv2.circle(mask, (int(p2[0]), int(p2[1])), 5, 255, -1)

    return cv2.medianBlur(mask, 7)


IMAGE SPLICING DETECTION


In [ ]:
def detect_splicing(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (11,11), 0)
    score, diff = ssim(gray, blur, full=True)
    diff = (1 - diff) * 255
    diff = diff.astype("uint8")
    return cv2.medianBlur(diff, 7)


WAVELET / FREQUENCY ARTIFACT DETECTION


In [ ]:
def detect_wavelet(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    cA, (cH,cV,cD) = pywt.dwt2(gray.astype("float32"), "haar")
    detail = np.abs(cH)+np.abs(cV)+np.abs(cD)
    detail = ((detail - detail.min())/(np.ptp(detail)+1e-9)*255).astype("uint8")
    detail = cv2.resize(detail, (gray.shape[1], gray.shape[0]))
    return cv2.medianBlur(detail, 7)

GENERATING HEATMAP

In [ ]:
def make_heatmap(mask, img):
    mask = cv2.GaussianBlur(mask.astype("float32"), (31,31), 0)
    mask = (mask / (mask.max()+1e-9) * 255).astype("uint8")
    heat = cv2.applyColorMap(mask, cv2.COLORMAP_JET)
    return cv2.addWeighted(img, 0.6, heat, 0.6, 0)


FINAL TAMPER DECISION LOGIC


In [ ]:
def get_decision(cm, sp, wv):
    total = cm.size

    score = (
        0.40 * (np.sum(cm > 127) / total) +
        0.30 * (np.sum(sp > 127) / total) +
        0.30 * (np.sum(wv > 127) / total)
    )

    print("\nTamper Score:", score)
    if score > 0.24:
        return "TAMPERED", score
    else:
        return "AUTHENTIC", score

**4TH REVIEW**

**USER INPUT**

User uploads image(s) directly through the Google Colab file upload interface.

Supported formats: JPG, PNG

No manual path entry is required.

Input Type:
✔ Single image
✔ Multiple images (batch processing supported)

USER DEFINED PATH

In [ ]:
def load_image(path):
    try:
        path = path.strip().strip('"').strip("'")

        if not os.path.isfile(path):
            print("❌ File does not exist at given path")
            return None

        img = Image.open(path).convert("RGB")
        img = np.array(img)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

        return img

    except Exception as e:
        print("❌ Error loading image:", e)
        return None


In [ ]:
from PIL import Image
from google.colab import files

def preprocess(img):
    # Only resize + light sharpening
    h, w = img.shape[:2]
    max_dim = 512

    if max(h, w) > max_dim:
        scale = max_dim / max(h, w)
        img = cv2.resize(img, (int(w*scale), int(h*scale)))

    # Light sharpening (keeps tampering details)
    kernel = np.array([[0,-1,0],
                       [-1,5,-1],
                       [0,-1,0]])
    img = cv2.filter2D(img, -1, kernel)

    return img

def detect_copy_move(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    orb = cv2.ORB_create(nfeatures=1500)
    kp, des = orb.detectAndCompute(gray, None)

    mask = np.zeros_like(gray)

    if des is None or len(kp) < 10:
        return mask

    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = bf.match(des, des)

    for m in matches:
        if m.queryIdx == m.trainIdx:
            continue
        if m.distance < 30:
            p1 = kp[m.queryIdx].pt
            p2 = kp[m.trainIdx].pt
            if abs(p1[0] - p2[0]) + abs(p1[1] - p2[1]) < 12:
                continue
            cv2.circle(mask, (int(p1[0]), int(p1[1])), 5, 255, -1)
            cv2.circle(mask, (int(p2[0]), int(p2[1])), 5, 255, -1)

    return cv2.medianBlur(mask, 7)

def detect_splicing(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (11,11), 0)
    score, diff = ssim(gray, blur, full=True)
    diff = (1 - diff) * 255
    diff = diff.astype("uint8")
    return cv2.medianBlur(diff, 7)


while True:
    print("\n📂 Upload image(s) or click Cancel to exit")
    uploaded = files.upload()

    if len(uploaded) == 0:
        print("Program ended.")
        break

    for filename in uploaded.keys():
        path = f"/content/{filename}"
        print(f"\n🔍 Processing: {path}")

        img = load_image(path)
        if img is None:
            print("Invalid image.")
            continue

        before = img.copy()
        after = preprocess(img)

        cm = detect_copy_move(after)
        sp = detect_splicing(after)
        wv = detect_wavelet(after)

        result, score = get_decision(cm, sp, wv)
        print("\nFINAL RESULT:", result)

        # ALWAYS show before + after
        show_image("Before Preprocessing", before)
        show_image("After Preprocessing", after)

        if result == "TAMPERED":
            mask = (np.maximum(cm, np.maximum(sp, wv)) > 127).astype("uint8") * 255
            heatmap = make_heatmap(mask, after)

            show_image("Binary Mask", mask)
            show_image("Heatmap", heatmap)


Output hidden; open in https://colab.research.google.com to view.

FINAL REVIEW

**CONCLUSION**

This project successfully demonstrates a classical image tampering detection system using image processing and forensic analysis techniques. By combining copy–move detection, splicing detection, and frequency-domain analysis, the system is able to detect manipulated images without the need for machine learning or large datasets. The addition of heatmap visualization enhances interpretability by clearly indicating suspected tampered regions.
The project is effective for academic purposes, digital forensics understanding, and forms a strong foundation for future enhancement using deep learning-based approaches.